# Batch 5 — Pre-Scan Summary



In [ ]:
import pandas as pd
from pathlib import Path
csv_path = Path(r'../data/batch5_pre_scan_summary.csv').resolve()
try:
    df = pd.read_csv(csv_path)
    df
except Exception:
    print('Failed to load', csv_path)


,filename,sex,reference_image,sample_area_cm2,bone_area_cm2,total_weight_g,soft_weight_g,lean_weight_g,fat_weight_g,fat_percent,BMC_g,BMD_mg_per_cm2
0,B8M0.txt,Male,B8M0,32.398,9.082,34.8332,34.2006,25.3143,8.8863,25.983,0.63268,69.660
1,B8M1.txt,Male,B8M1,31.003,7.825,34.3067,33.7900,26.8146,6.9754,20.643,0.51672,66.032
2,B8M2.txt,Male,B8M2,35.079,9.292,38.7894,38.1942,23.3746,14.8196,38.801,0.59517,64.050
3,B8M3.txt,Male,B8M3,27.402,7.496,28.0828,27.6151,19.8172,7.7980,28.238,0.46764,62.383
4,B8M4.txt,Male,B8M4,27.802,8.608,28.4824,27.8666,20.3151,7.5515,27.099,0.61585,71.548
5,B8M5.txt,Male,B8M5,28.831,8.034,29.3442,28.8112,20.0824,8.7288,30.297,0.53303,66.350
6,B8F0.txt,Female,B8F0,30.514,9.659,30.1544,29.5046,16.9172,12.5873,42.662,0.64984,67.276
7,B8F1.txt,Female,B8F1,32.262,9.754,33.4909,32.8118,19.1544,13.6573,41.623,0.67914,69.626
8,B8F2.txt,Female,B8F2,24.608,8.177,20.7063,20.1771,14.2245,5.9526,29.502,0.52917,64.717
9,B8F3.txt,Female,B8F3,31.877,10.085,33.9274,33.3042,16.1373,17.1670,51.546,0.62315,61.787


## Batch 5 — 1-week post-treatment summary



In [ ]:
# Load week-1 master CSV if present, otherwise scan for week-1 TXT files and build one; then display rows for this batch
from pathlib import Path
import re

DATA_CSV = Path(r'../data/week1_reports.csv').resolve()
batch_name = 'Batch 5'

def scan_and_build(downloads_root):
    week1_re = re.compile(r'(?:week[\s_-]*1|1[\s_-]*week|wk[\s_-]*1|week1|1week)', re.I)
    patterns = {
        'sample_area': re.compile(r"Sample Area:\s*([0-9.]+)\s*cm\^2"),
        'bone_area': re.compile(r"Bone Area:\s*([0-9.]+)\s*cm\^2"),
        'total_weight': re.compile(r"Total Weight:\s*([0-9.]+)\s*g"),
        'soft_weight': re.compile(r"Soft Weight:\s*([0-9.]+)\s*g"),
        'lean_weight': re.compile(r"Lean Weight:\s*([0-9.]+)\s*g"),
        'fat_weight': re.compile(r"Fat Weight:\s*([0-9.]+)\s*g"),
        'fat_percent': re.compile(r"Fat Percent:\s*([0-9.]+)"),
        'BMC': re.compile(r"BMC:\s*([0-9.]+)\s*g"),
        'BMD': re.compile(r"BMD:\s*([0-9.]+)\s*mg/cm\^2"),
    }
    rows = []
    for txt in downloads_root.rglob('*.txt'):
        nl = str(txt).lower()
        if week1_re.search(nl):
            text = txt.read_text(encoding='utf-8', errors='replace')
            inside_block = ''
            whole_block = ''
            if 'INSIDE ROI TISSUE STATISTICS:' in text and 'WHOLE TISSUE STATISTICS:' in text:
                inside_block = text.split('INSIDE ROI TISSUE STATISTICS:')[1].split('WHOLE TISSUE STATISTICS:')[0]
                whole_block = text.split('WHOLE TISSUE STATISTICS:')[1]
            else:
                inside_block = text
            parts = txt.parts
            batch = next((p for p in parts if p.lower().startswith('batch')), '')
            sex = 'Male' if any(p.lower()=='male' for p in parts) else ('Female' if any(p.lower()=='female' for p in parts) else '')
            row = {'batch': batch, 'sex': sex, 'filename': txt.name}
            for k,p in patterns.items():
                mi = p.search(inside_block)
                mw = p.search(whole_block)
                row[f'inside_{k}'] = mi.group(1) if mi else ''
                row[f'whole_{k}'] = mw.group(1) if mw else ''
            rows.append(row)
    return rows

try:
    import pandas as pd
    if DATA_CSV.exists():
        master = pd.read_csv(DATA_CSV)
    else:
        downloads_root = Path(r"c:\Users\hanss\Downloads\DEXA Scans (1)\DEXA Scans")
        rows = scan_and_build(downloads_root)
        master = pd.DataFrame(rows)
        try:
            master.to_csv(DATA_CSV, index=False)
        except Exception:
            pass
    if not master.empty:
        df_batch = master[master['batch'].str.lower()==batch_name.lower()]
        if not df_batch.empty:
            display(df_batch.reset_index(drop=True))
        else:
            print('No week-1 rows for', batch_name)
    else:
        print('No week-1 files found anywhere')
except Exception as e:
    print('Error building/displaying week-1 table:', e)


,batch,sex,filename,path,inside_sample_area,whole_sample_area,inside_bone_area,whole_bone_area,inside_total_weight,whole_total_weight,...,inside_lean_weight,whole_lean_weight,inside_fat_weight,whole_fat_weight,inside_fat_percent,whole_fat_percent,inside_BMC,whole_BMC,inside_BMD,whole_BMD
0,Batch 5,Female,B8F0.txt,c:\Users\hanss\Downloads\DEXA Scans (1)\DEXA S...,27.763,29.213,8.308,9.220,27.1631,29.1435,...,18.2298,19.1049,8.3790,9.3321,31.490,32.817,0.55424,0.70652,66.709,76.631
1,Batch 5,Female,B8F1.txt,c:\Users\hanss\Downloads\DEXA Scans (1)\DEXA S...,30.863,33.204,10.129,11.422,32.6958,35.5124,...,19.8387,20.9668,12.1484,13.6394,37.979,39.413,0.70874,0.90617,69.969,79.335
2,Batch 5,Female,B8F2.txt,c:\Users\hanss\Downloads\DEXA Scans (1)\DEXA S...,24.485,26.061,8.425,9.406,21.1156,23.1137,...,14.3057,15.3662,6.2490,7.0661,30.402,31.499,0.56086,0.68144,66.570,72.451
3,Batch 5,Female,B8F3.txt,c:\Users\hanss\Downloads\DEXA Scans (1)\DEXA S...,30.677,33.006,9.494,10.735,33.3120,36.0837,...,16.6685,17.6179,16.0460,17.6976,49.049,50.113,0.59750,0.76811,62.931,71.551
4,Batch 5,Male,B8M0.txt,c:\Users\hanss\Downloads\DEXA Scans (1)\DEXA S...,28.117,29.902,8.452,9.260,28.8994,31.2029,...,22.2282,23.7347,6.0874,6.7645,21.498,22.179,0.58385,0.70373,69.076,76.000
5,Batch 5,Male,B8M1.txt,c:\Users\hanss\Downloads\DEXA Scans (1)\DEXA S...,30.551,33.163,8.286,9.522,33.7242,37.1166,...,25.9647,28.3131,7.2090,8.1062,21.731,22.258,0.55050,0.69736,66.433,73.234
6,Batch 5,Male,B8M2.txt,c:\Users\hanss\Downloads\DEXA Scans (1)\DEXA S...,31.493,34.233,9.267,10.322,36.8304,39.5304,...,22.4274,23.6475,13.8207,15.1744,38.128,39.087,0.58231,0.70852,62.840,68.639
7,Batch 5,Male,B8M3.txt,c:\Users\hanss\Downloads\DEXA Scans (1)\DEXA S...,26.865,29.677,7.392,8.704,26.5886,29.5126,...,18.5441,20.0461,7.5538,8.7841,28.944,30.468,0.49075,0.68233,66.387,78.396
8,Batch 5,Male,B8M4.txt,c:\Users\hanss\Downloads\DEXA Scans (1)\DEXA S...,27.700,30.295,8.358,9.777,28.4284,31.6898,...,21.4371,23.2638,6.4050,7.6329,23.005,24.705,0.58624,0.79303,70.141,81.115
9,Batch 5,Male,B8M5.txt,c:\Users\hanss\Downloads\DEXA Scans (1)\DEXA S...,27.040,29.643,7.589,8.637,26.9360,29.4516,...,19.3388,20.9549,7.1122,7.8683,26.888,27.299,0.48500,0.62845,63.906,72.760


## Batch 5 — 2-week post-treatment summary



In [2]:
# Batch 5 — 2-week post-treatment summary
# Scan/build week-2 master CSV and display rows for this batch
from pathlib import Path
import re

DATA_CSV2 = Path(r'../data/week2_reports.csv').resolve()
batch_name = 'Batch 5'

def scan_and_build_week(downloads_root, week_num):
    week_re = re.compile(rf'(?:week[\s_-]*{week_num}|{week_num}[\s_-]*week|wk[\s_-]*{week_num}|week{week_num}|{week_num}week)', re.I)
    patterns = {
        'sample_area': re.compile(r"Sample Area:\s*([0-9.]+)\s*cm\^2"),
        'bone_area': re.compile(r"Bone Area:\s*([0-9.]+)\s*cm\^2"),
        'total_weight': re.compile(r"Total Weight:\s*([0-9.]+)\s*g"),
        'soft_weight': re.compile(r"Soft Weight:\s*([0-9.]+)\s*g"),
        'lean_weight': re.compile(r"Lean Weight:\s*([0-9.]+)\s*g"),
        'fat_weight': re.compile(r"Fat Weight:\s*([0-9.]+)\s*g"),
        'fat_percent': re.compile(r"Fat Percent:\s*([0-9.]+)"),
        'BMC': re.compile(r"BMC:\s*([0-9.]+)\s*g"),
        'BMD': re.compile(r"BMD:\s*([0-9.]+)\s*mg/cm\^2"),
    }
    rows = []
    for txt in downloads_root.rglob('*.txt'):
        nl = str(txt).lower()
        if week_re.search(nl):
            text = txt.read_text(encoding='utf-8', errors='replace')
            inside_block = ''
            whole_block = ''
            if 'INSIDE ROI TISSUE STATISTICS:' in text and 'WHOLE TISSUE STATISTICS:' in text:
                inside_block = text.split('INSIDE ROI TISSUE STATISTICS:')[1].split('WHOLE TISSUE STATISTICS:')[0]
                whole_block = text.split('WHOLE TISSUE STATISTICS:')[1]
            else:
                inside_block = text
            parts = txt.parts
            batch = next((p for p in parts if p.lower().startswith('batch')), '')
            sex = 'Male' if any(p.lower()=='male' for p in parts) else ('Female' if any(p.lower()=='female' for p in parts) else '')
            row = {'batch': batch, 'sex': sex, 'filename': txt.name}
            for k,p in patterns.items():
                mi = p.search(inside_block)
                mw = p.search(whole_block)
                row[f'inside_{k}'] = mi.group(1) if mi else ''
                row[f'whole_{k}'] = mw.group(1) if mw else ''
            rows.append(row)
    return rows

try:
    import pandas as pd
    if DATA_CSV2.exists():
        master2 = pd.read_csv(DATA_CSV2)
    else:
        downloads_root = Path(r"c:\Users\hanss\Downloads\DEXA Scans (1)\DEXA Scans")
        rows2 = scan_and_build_week(downloads_root, 2)
        master2 = pd.DataFrame(rows2)
        try:
            master2.to_csv(DATA_CSV2, index=False)
        except Exception:
            pass
    if not master2.empty:
        df_batch2 = master2[master2['batch'].str.lower()==batch_name.lower()]
        if not df_batch2.empty:
            display(df_batch2.reset_index(drop=True))
        else:
            print('No week-2 rows for', batch_name)
    else:
        print('No week-2 files found anywhere')
except Exception as e:
    print('Error building/displaying week-2 table:', e)


,batch,sex,filename,inside_sample_area,whole_sample_area,inside_bone_area,whole_bone_area,inside_total_weight,whole_total_weight,inside_soft_weight,...,inside_lean_weight,whole_lean_weight,inside_fat_weight,whole_fat_weight,inside_fat_percent,whole_fat_percent,inside_BMC,whole_BMC,inside_BMD,whole_BMD
0,Batch 5,Female,B8F0.txt,26.212,31.693,8.826,13.302,23.3461,29.4178,22.7211,...,16.4107,18.4000,6.3104,9.1640,27.774,33.246,0.62497,1.85383,70.809,139.369
1,Batch 5,Female,B8F1.txt,29.719,35.691,10.065,15.050,32.5595,39.3758,31.7556,...,19.7378,21.4890,12.0178,15.7831,37.845,42.346,0.80398,2.10368,79.875,139.776
2,Batch 5,Female,B8F2.txt,24.416,30.130,8.526,13.291,21.6130,27.4373,20.9897,...,15.1849,17.5402,5.8048,8.0349,27.656,31.417,0.62328,1.86222,73.100,140.114
3,Batch 5,Female,B8F3.txt,30.175,35.953,9.615,14.261,31.8302,37.9845,31.2058,...,17.1271,18.9167,14.0787,17.2256,45.116,47.660,0.62439,1.84216,64.938,129.178
4,Batch 5,Male,B8M0.txt,28.849,35.072,8.964,14.015,30.8714,37.5405,30.1787,...,22.4812,24.4668,7.6974,11.1323,25.506,31.271,0.69275,1.94137,77.283,138.525
5,Batch 5,Male,B8M1.txt,29.871,36.305,8.379,13.290,33.4092,39.8175,32.8288,...,25.6117,29.3019,7.2171,8.7971,21.984,23.090,0.58039,1.71842,69.264,129.299
6,Batch 5,Male,B8M2.txt,32.188,39.193,8.824,14.005,32.6131,38.8788,32.0303,...,21.1179,23.4970,10.9124,13.6299,34.069,36.712,0.58284,1.75183,66.052,125.089
7,Batch 5,Male,B8M3.txt,25.653,31.798,7.514,12.201,24.1735,30.4664,23.6753,...,17.2242,19.9191,6.4510,8.9000,27.248,30.882,0.49829,1.64735,66.318,135.018
8,Batch 5,Male,B8M4.txt,27.965,34.311,8.376,13.152,27.6899,34.0603,27.0907,...,21.3545,24.2150,5.7362,8.0914,21.174,25.046,0.59922,1.75393,71.535,133.357
9,Batch 5,Male,B8M5.txt,28.313,34.718,7.597,12.627,27.8845,34.0323,27.3821,...,20.8380,23.5553,6.5441,8.8270,23.899,27.259,0.50243,1.65001,66.137,130.676


## Batch 5 — 3-week post-treatment summary



In [1]:
# Batch 5 — 3-week post-treatment summary
# Scan/build week-3 master CSV and display rows for this batch (same logic as week-2)
from pathlib import Path
import re

DATA_CSV3 = Path(r'../data/week3_reports.csv').resolve()
batch_name = 'Batch 5'

try:
    import pandas as pd
    if DATA_CSV3.exists():
        master3 = pd.read_csv(DATA_CSV3)
    else:
        downloads_root = Path(r"c:\Users\hanss\Downloads\DEXA Scans (1)\DEXA Scans")
        rows3 = scan_and_build_week(downloads_root, 3)
        master3 = pd.DataFrame(rows3)
        try:
            master3.to_csv(DATA_CSV3, index=False)
        except Exception:
            pass
    if not master3.empty:
        df_batch3 = master3[master3['batch'].str.lower()==batch_name.lower()]
        if not df_batch3.empty:
            display(df_batch3.reset_index(drop=True))
        else:
            print('No week-3 rows for', batch_name)
    else:
        print('No week-3 files found anywhere')
except Exception as e:
    print('Error building/displaying week-3 table:', e)


,batch,sex,filename,inside_sample_area,whole_sample_area,inside_bone_area,whole_bone_area,inside_total_weight,whole_total_weight,inside_soft_weight,...,inside_lean_weight,whole_lean_weight,inside_fat_weight,whole_fat_weight,inside_fat_percent,whole_fat_percent,inside_BMC,whole_BMC,inside_BMD,whole_BMD
0,Batch 5,Female,B8F0.txt,25.828,31.673,9.075,14.010,24.9138,30.4224,24.2357,...,17.6253,20.1999,6.6104,8.2632,27.275,29.031,0.67809,1.95926,74.718,139.850
1,Batch 5,Female,B8F1.txt,31.371,33.155,10.550,11.359,32.9969,34.7588,32.0825,...,19.5828,20.2492,12.4997,13.4879,38.961,39.979,0.91440,1.02168,86.669,89.948
2,Batch 5,Female,B8F2.txt,23.527,24.435,8.527,9.104,21.0758,22.1851,20.4012,...,14.4920,15.1355,5.9092,6.2985,28.965,29.386,0.67468,0.75109,79.124,82.497
3,Batch 5,Female,B8F3.txt,30.296,31.827,9.001,9.771,30.4451,32.0830,29.8935,...,17.7346,18.4249,12.1589,13.0304,40.674,41.425,0.55158,0.62763,61.280,64.231
4,Batch 5,Male,B8M0.txt,29.029,31.047,9.193,10.346,30.6655,33.4972,29.9543,...,22.6942,23.9523,7.2601,8.6167,24.237,26.457,0.71120,0.92814,77.366,89.711
5,Batch 5,Male,B8M1.txt,30.196,32.083,9.308,10.404,32.6571,35.2326,32.0104,...,25.3349,26.5701,6.6755,7.8360,20.854,22.775,0.64675,0.82652,69.486,79.441
6,Batch 5,Male,B8M2.txt,31.178,32.603,8.830,9.360,31.5192,33.0357,30.9635,...,21.8849,22.8053,9.0787,9.6260,29.321,29.681,0.55567,0.60432,62.930,64.567
7,Batch 5,Male,B8M3.txt,26.767,28.816,7.461,8.619,25.9190,28.7023,25.4087,...,18.5563,20.2088,6.8524,7.8023,26.969,27.854,0.51032,0.69121,68.400,80.196
8,Batch 5,Male,B8M4.txt,26.681,28.175,8.245,8.879,27.6430,29.2476,27.0417,...,20.2951,21.1889,6.7467,7.3954,24.949,25.872,0.60125,0.66334,72.920,74.711
9,Batch 5,Male,B8M5.txt,25.570,27.106,6.980,7.690,24.0830,25.9439,23.5957,...,18.2436,19.5778,5.3521,5.8034,22.683,22.865,0.48722,0.56273,69.803,73.176
